# NBA MVP Project — Web Scraping (Rate-Limited + Resume)

This notebook follows the CS229 reference project's NBA MVP setup.

It collects the **10 statistical features**:
`G, MP, FG%, 3P%, FT%, TRB, AST, STL, BLK, PTS`

It also collects:
`Year, Rank, Player, MVP`

The scraper is deliberately rate-limited and saves progress after each season so that an interrupted run does not lose completed work.


In [3]:
import os
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1200)

# IMPORTANT:
# Basketball-Reference rate-limits automated requests.
# This notebook uses a slow request interval and does NOT try to bypass
# a 403 block with proxies, header spoofing, or other evasion methods.

REQUEST_DELAY = 5.0
MAX_RETRIES = 2

BASE_DIR = os.getcwd()
PROGRESS_FILE = os.path.join(BASE_DIR, "NBA_MVP_progress.csv")
FINAL_FILE = os.path.join(BASE_DIR, "NBA_MVP_10_Features.csv")

training_years = list(range(2005, 2021))
prediction_year = 2021
years = training_years + [prediction_year]

feature_columns = [
    "G", "MP", "FG%", "3P%", "FT%",
    "TRB", "AST", "STL", "BLK", "PTS"
]

print("Training years:", training_years)
print("Prediction year:", prediction_year)
print("Total seasons:", len(years))


Training years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Prediction year: 2021
Total seasons: 17


In [4]:
# A single persistent session is used for all requests.
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

last_request_time = 0.0

def polite_get(url):
    global last_request_time

    # Keep requests well below Sports Reference's published rate limit.
    elapsed = time.monotonic() - last_request_time
    if elapsed < REQUEST_DELAY:
        time.sleep(REQUEST_DELAY - elapsed)

    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(url, timeout=30)
            last_request_time = time.monotonic()

            if response.status_code == 403:
                print("\n403 Forbidden received.")
                print("Basketball-Reference has blocked/rate-limited this session.")
                print("Do NOT keep rerunning the notebook repeatedly.")
                print("Wait until the block expires, then rerun/resume.")
                return None

            response.raise_for_status()
            return response

        except requests.RequestException as e:
            if attempt < MAX_RETRIES:
                wait = REQUEST_DELAY * (attempt + 2)
                print(f"Request failed: {e}")
                print(f"Waiting {wait:.0f}s before retry...")
                time.sleep(wait)
            else:
                print(f"Request failed after retries: {e}")
                return None

    return None


In [5]:
def scrape_top_mvp_candidates(year):
    url = f"https://www.basketball-reference.com/awards/awards_{year}.html"
    response = polite_get(url)

    if response is None:
        return None

    tables = pd.read_html(response.text)
    mvp_df = None

    for table in tables:
        cols = set(table.columns)

        if {"Player", "Share"}.issubset(cols) and "Pts Won" in cols:
            mvp_df = table.copy()
            break

    if mvp_df is None:
        raise ValueError(f"MVP table not found for {year}")

    mvp_df = mvp_df.dropna(subset=["Player"]).copy()

    # The paper uses the top three MVP candidates.
    mvp_df = mvp_df.head(3).copy()

    mvp_df["Rank"] = range(1, len(mvp_df) + 1)
    mvp_df["MVP"] = (mvp_df["Rank"] == 1).astype(int)
    mvp_df["Year"] = year

    return mvp_df[["Rank", "Player", "Year", "MVP"]]


In [6]:
def scrape_player_stats(year):
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html"
    response = polite_get(url)

    if response is None:
        return None

    tables = pd.read_html(response.text)

    required = {
        "Player", "G", "MP", "FG%", "3P%", "FT%",
        "TRB", "AST", "STL", "BLK", "PTS"
    }

    stats_df = None

    for table in tables:
        if required.issubset(set(table.columns)):
            stats_df = table.copy()
            break

    if stats_df is None:
        raise ValueError(f"Per-game statistics table not found for {year}")

    stats_df = stats_df[
        [
            "Player", "G", "MP", "FG%", "3P%", "FT%",
            "TRB", "AST", "STL", "BLK", "PTS"
        ]
    ].copy()

    stats_df["Year"] = year

    # Remove repeated header rows / invalid player rows.
    stats_df = stats_df[
        stats_df["Player"].notna() &
        (stats_df["Player"] != "Player")
    ].copy()

    return stats_df


In [7]:
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        df = pd.read_csv(PROGRESS_FILE)

        if not df.empty:
            print(f"Loaded existing progress: {len(df)} rows")
            return df

    return pd.DataFrame()

def save_progress(df):
    df.to_csv(PROGRESS_FILE, index=False)
    print(f"Progress saved -> {PROGRESS_FILE}")


In [8]:
# STEP 1: Get the top three MVP candidates for each season.
#
# The loop stops cleanly if Basketball-Reference returns 403.
# Completed seasons are saved to disk.

mvp_candidates = load_progress()

if mvp_candidates.empty:
    mvp_candidates = pd.DataFrame(
        columns=["Rank", "Player", "Year", "MVP"]
    )

completed_mvp_years = set(
    pd.to_numeric(mvp_candidates["Year"], errors="coerce")
    .dropna()
    .astype(int)
    .tolist()
)

for year in years:
    if year in completed_mvp_years:
        print(f"Skipping {year} — already saved.")
        continue

    print(f"\nScraping MVP candidates for {year}...")

    try:
        result = scrape_top_mvp_candidates(year)

        if result is None:
            print("Stopping because the site returned 403 or the request failed.")
            break

        mvp_candidates = pd.concat(
            [mvp_candidates, result],
            ignore_index=True
        )

        save_progress(mvp_candidates)

    except Exception as e:
        print(f"Error in {year}: {e}")
        print("Stopping so the saved progress is not damaged.")
        break

mvp_candidates



Scraping MVP candidates for 2005...

403 Forbidden received.
Basketball-Reference has blocked/rate-limited this session.
Do NOT keep rerunning the notebook repeatedly.
Wait until the block expires, then rerun/resume.
Stopping because the site returned 403 or the request failed.


,Rank,Player,Year,MVP


In [ ]:
# STEP 2: Get the per-game statistics for the same seasons.
#
# This is a separate progress file so that the notebook can resume.

STATS_PROGRESS_FILE = os.path.join(BASE_DIR, "NBA_player_stats_progress.csv")

def load_stats_progress():
    if os.path.exists(STATS_PROGRESS_FILE):
        df = pd.read_csv(STATS_PROGRESS_FILE)

        if not df.empty:
            print(f"Loaded existing statistics progress: {len(df)} rows")
            return df

    return pd.DataFrame()

def save_stats_progress(df):
    df.to_csv(STATS_PROGRESS_FILE, index=False)
    print(f"Statistics progress saved -> {STATS_PROGRESS_FILE}")

player_stats = load_stats_progress()

if player_stats.empty:
    player_stats = pd.DataFrame()

completed_stats_years = set()

if not player_stats.empty and "Year" in player_stats.columns:
    completed_stats_years = set(
        pd.to_numeric(player_stats["Year"], errors="coerce")
        .dropna()
        .astype(int)
        .tolist()
    )

for year in years:
    if year in completed_stats_years:
        print(f"Skipping statistics for {year} — already saved.")
        continue

    print(f"\nScraping player statistics for {year}...")

    try:
        result = scrape_player_stats(year)

        if result is None:
            print("Stopping because the site returned 403 or the request failed.")
            break

        player_stats = pd.concat(
            [player_stats, result],
            ignore_index=True
        )

        save_stats_progress(player_stats)

    except Exception as e:
        print(f"Error in {year}: {e}")
        print("Stopping so the saved progress is not damaged.")
        break

player_stats.head()


In [ ]:
# STEP 3: Merge the MVP candidates with their season statistics.

if mvp_candidates.empty:
    raise ValueError(
        "No MVP candidate data was collected. "
        "Check the saved progress file and the Basketball-Reference access status."
    )

if player_stats.empty:
    raise ValueError(
        "No player statistics were collected. "
        "Check the saved statistics progress file and the Basketball-Reference access status."
    )

final_data = pd.merge(
    mvp_candidates,
    player_stats,
    on=["Player", "Year"],
    how="left"
)

final_data = final_data[
    ["Year", "Rank", "Player"] + feature_columns + ["MVP"]
].copy()

for col in feature_columns:
    final_data[col] = pd.to_numeric(
        final_data[col],
        errors="coerce"
    )

final_data["MVP"] = pd.to_numeric(
    final_data["MVP"],
    errors="coerce"
)

# Only remove rows where one of the required 10 features is missing.
final_data = final_data.dropna(
    subset=feature_columns
).reset_index(drop=True)

final_data


In [ ]:
# STEP 4: Verify that every feature required by the PDF is present.

print("Columns in final dataset:")
print(list(final_data.columns))

print("\nRequired 10 ML features:")
for feature in feature_columns:
    print(f"✓ {feature}")

print(f"\nFinal dataset shape: {final_data.shape}")
print(f"Number of seasons: {final_data['Year'].nunique()}")
print(f"Number of players: {len(final_data)}")

print("\nMVP label counts:")
print(final_data["MVP"].value_counts().sort_index())


In [ ]:
# STEP 5: Save the final CSV.

final_data.to_csv(
    FINAL_FILE,
    index=False
)

print(f"\nFinal dataset saved to:")
print(FINAL_FILE)


In [ ]:
# Optional: remove progress files AFTER you have verified the final CSV.
#
# DO NOT run this cell until you are completely satisfied with the data.
#
# os.remove(PROGRESS_FILE)
# os.remove(STATS_PROGRESS_FILE)
# print("Progress files removed.")
